# Canonical two-qutrit Bell baseline

This notebook prepares a deterministic canonical direct-basis input for the two-qutrit Bell experiment. It does not submit jobs or use credentials; runtime results belong in `artifacts/`.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "qudits_on_qubits").is_dir():
            return candidate
    raise RuntimeError(
        "Cannot find repository root. Start from this repository or a descendant "
        "containing pyproject.toml and src/qudits_on_qubits."
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment
from qudits_on_qubits.benchmarks.direct_basis.circuits import build_direct_basis_graph_state_circuit
from qudits_on_qubits.experiments import (
    AerIdeal,
    BootstrapConfig,
    ExperimentSpec,
    IQMHardware,
    MitigationConfig,
    PathBasis,
    PiastQHardware,
    run_experiment,
)

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_single_circuit(path):
    try:
        with Path(path).open("rb") as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f"unable to load canonical basis QPY: {path}") from error
    if len(circuits) != 1:
        raise RuntimeError(f"canonical basis QPY must contain exactly one circuit, found {len(circuits)}")
    return circuits[0]


def validate_canonical_basis(directory, expected_encoding, expected_circuit):
    directory = Path(directory)
    required_files = {"graph_state_direct_basis.qpy", "E.npy", "metadata.json"}
    try:
        actual_files = {path.name for path in directory.iterdir()}
    except OSError as error:
        raise RuntimeError(f"canonical basis directory is unavailable: {directory}") from error
    if actual_files != required_files:
        raise RuntimeError(
            "canonical basis files must be exactly graph_state_direct_basis.qpy, E.npy, "
            f"and metadata.json; found {sorted(actual_files)}"
        )

    encoding_path = directory / "E.npy"
    try:
        encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(f"canonical basis encoding is invalid: {encoding_path}") from error
    if encoding.shape != (4, 3):
        raise RuntimeError(f"canonical basis encoding must have shape (4, 3), got {encoding.shape}")
    try:
        is_finite = np.isfinite(encoding).all()
    except TypeError as error:
        raise RuntimeError("canonical basis encoding must be numeric and finite") from error
    if not is_finite:
        raise RuntimeError("canonical basis encoding must be finite")
    if not np.allclose(encoding.conj().T @ encoding, np.eye(3), atol=1e-12, rtol=0):
        raise RuntimeError("canonical basis encoding must be an isometry")
    if not np.array_equal(encoding, expected_encoding):
        raise RuntimeError("canonical basis encoding does not match canonical_ez")

    circuit_path = directory / "graph_state_direct_basis.qpy"
    circuit = load_single_circuit(circuit_path)
    if circuit.num_qubits != 4 or circuit.num_clbits != 0:
        raise RuntimeError("canonical basis QPY must contain one unmeasured four-qubit circuit")
    for instruction in circuit.data:
        operation = instruction.operation
        if operation.name in {"measure", "reset"}:
            raise RuntimeError("canonical basis QPY must not contain measurements or resets")
        if getattr(operation, "condition", None) is not None:
            raise RuntimeError("canonical basis QPY must not contain conditioned instructions")
        if getattr(operation, "blocks", ()):
            raise RuntimeError("canonical basis QPY must not contain control flow")
    try:
        same_state = Statevector.from_instruction(circuit).equiv(
            Statevector.from_instruction(expected_circuit)
        )
    except Exception as error:
        raise RuntimeError("canonical basis QPY circuit cannot be validated as a state preparation") from error
    if not same_state:
        raise RuntimeError("canonical basis QPY circuit does not match the canonical graph state")

    metadata_path = directory / "metadata.json"
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(f"canonical basis metadata is invalid: {metadata_path}") from error
    expected_metadata = {
        "schema": "qoq-reference-basis-v1",
        "state": "two_qutrit",
        "encoding_id": "canonical_ez",
        "num_qubits": 4,
        "encoding_shape": [4, 3],
        "files": {
            "graph_state_direct_basis.qpy": {"sha256": sha256_file(circuit_path)},
            "E.npy": {"sha256": sha256_file(encoding_path)},
        },
    }
    if metadata != expected_metadata:
        raise RuntimeError("canonical basis metadata does not match the validated bundle")


def prepare_canonical_basis(repo_root):
    repo_root = Path(repo_root)
    expected_encoding = get_encoding("canonical_ez").as_array()
    expected_circuit = build_direct_basis_graph_state_circuit("two_qutrit", expected_encoding)
    directory = repo_root / "experiment_inputs" / "reference_bases" / "two_qutrit" / "canonical_ez"

    if directory.exists():
        validate_canonical_basis(directory, expected_encoding, expected_circuit)
        return directory

    directory.mkdir(parents=True, exist_ok=False)
    temporary_files = []
    try:
        qpy_temp = directory / f".graph_state_direct_basis.{uuid4().hex}.qpy.tmp"
        encoding_temp = directory / f".E.{uuid4().hex}.npy.tmp"
        metadata_temp = directory / f".metadata.{uuid4().hex}.json.tmp"
        temporary_files.extend((qpy_temp, encoding_temp, metadata_temp))

        with qpy_temp.open("wb") as handle:
            qpy.dump(expected_circuit, handle)
        with encoding_temp.open("wb") as handle:
            np.save(handle, expected_encoding, allow_pickle=False)
        metadata = {
            "schema": "qoq-reference-basis-v1",
            "state": "two_qutrit",
            "encoding_id": "canonical_ez",
            "num_qubits": 4,
            "encoding_shape": [4, 3],
            "files": {
                "graph_state_direct_basis.qpy": {"sha256": sha256_file(qpy_temp)},
                "E.npy": {"sha256": sha256_file(encoding_temp)},
            },
        }
        metadata_temp.write_text(
            json.dumps(metadata, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )

        qpy_temp.replace(directory / "graph_state_direct_basis.qpy")
        encoding_temp.replace(directory / "E.npy")
        metadata_temp.replace(directory / "metadata.json")
    finally:
        for temporary_file in temporary_files:
            if temporary_file.exists():
                temporary_file.unlink()

    validate_canonical_basis(directory, expected_encoding, expected_circuit)
    return directory

In [ ]:
CANONICAL_BASIS_DIRECTORY = prepare_canonical_basis(REPO_ROOT)
CANONICAL_BASIS_DIRECTORY